# Chapter 5: Automated Evaluators
Code-based checks, an LLM-as-judge, testing the judge (TPR / TNR), the corrected success rate with a confidence interval, and group metrics.

In [ ]:
from common import ask, embed, load_data
import json, random
import numpy as np

data = load_data()["ch5"]

## 1. Code-based evaluators
Fast, free, same answer every time. Use when the rule is objective.

In [ ]:
def valid_json(text):
    try:
        json.loads(text); return True
    except json.JSONDecodeError:
        return False

def exactly_3_bullets(text):
    try:
        return len(json.loads(text)["bullets"]) == 3
    except Exception:
        return False

for out in data["code_eval_outputs"]:
    print(f"valid_json={valid_json(out)!s:5}  3_bullets={exactly_3_bullets(out)!s:5}  {out[:50]}")

## 2. Split the labeled data: train / dev / test
- **train**: examples you may put inside the judge prompt
- **dev**: used to improve the prompt
- **test**: used once at the end (never look before)

In [ ]:
rng = random.Random(42)
pass_items, fail_items = data["pass_items"][:], data["fail_items"][:]
rng.shuffle(pass_items); rng.shuffle(fail_items)

train = pass_items[:2] + fail_items[:2]
dev   = pass_items[2:8] + fail_items[2:9]
test  = pass_items[8:] + fail_items[9:]

print("train:", len(train), "| dev:", len(dev), "| test:", len(test))

## 3. Build the LLM judge
Four parts: **one task**, Pass/Fail definitions, few-shot examples (from *train* only), JSON output.

In [ ]:
J = data["judge"]

def build_judge_prompt(text):
    shots = "\n".join(f'Message: "{x["text"]}"\nAnswer: {x["human_label"]}\n' for x in train)
    return f"""{J["task"]}

PASS means: {J["pass_definition"]}
FAIL means: {J["fail_definition"]}

Examples:
{shots}
Now judge this message:
Message: "{text}"

Reply as JSON: {{"reasoning": "one short sentence", "answer": "Pass" or "Fail"}}"""

def judge(text):
    out = ask(build_judge_prompt(text), as_json=True)
    return 1 if out["answer"].strip().lower() == "pass" else 0     # 1 = Pass, 0 = Fail

print(build_judge_prompt("Hey dude, nice house lol!"))
print("\nJudge says (1=Pass, 0=Fail):", judge("Hey dude, nice house lol!"))

## 4. Test the judge on the dev set: TPR and TNR
- **TPR**: of the real Passes, how many did the judge call Pass?
- **TNR**: of the real Fails, how many did the judge call Fail?

In [ ]:
def to_int(label):
    return 1 if label == "Pass" else 0

def tpr_tnr(human, preds):
    human, preds = np.array(human), np.array(preds)
    tpr = ((human == 1) & (preds == 1)).sum() / max((human == 1).sum(), 1)
    tnr = ((human == 0) & (preds == 0)).sum() / max((human == 0).sum(), 1)
    return tpr, tnr

dev_human = [to_int(x["human_label"]) for x in dev]
dev_preds = [judge(x["text"]) for x in dev]

tpr, tnr = tpr_tnr(dev_human, dev_preds)
print(f"DEV  TPR={tpr:.0%}  TNR={tnr:.0%}   (goal: both above ~90%)")

In [ ]:
# Look at the mistakes -> improve the prompt / definitions -> run again
for x, h, p in zip(dev, dev_human, dev_preds):
    if h != p:
        print(f"human={h} judge={p} | {x['text']}")

## 5. Final score on the test set (run once)

In [ ]:
test_human = [to_int(x["human_label"]) for x in test]
test_preds = [judge(x["text"]) for x in test]
tpr, tnr = tpr_tnr(test_human, test_preds)
print(f"TEST TPR={tpr:.0%}  TNR={tnr:.0%}")

## 6. True success rate + 95% confidence interval
The judge makes mistakes, so we **correct** its Pass rate:

`theta = (p_obs + TNR - 1) / (TPR + TNR - 1)`

The **bootstrap** re-draws the data many times to see how much the answer moves.

In [ ]:
def estimate_success_rate(test_labels, test_preds, unlabeled_preds, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    tl, tp, up = map(np.array, (test_labels, test_preds, unlabeled_preds))

    def theta_hat(tl, tp, up):
        tpr = ((tl == 1) & (tp == 1)).sum() / max((tl == 1).sum(), 1)
        tnr = ((tl == 0) & (tp == 0)).sum() / max((tl == 0).sum(), 1)
        if tpr + tnr - 1 <= 1e-9:
            return np.nan                                 # judge is no better than a coin flip
        return float(np.clip((up.mean() + tnr - 1) / (tpr + tnr - 1), 0, 1))

    theta = theta_hat(tl, tp, up)
    boots = []
    for _ in range(B):
        i = rng.integers(0, len(tl), len(tl))             # resample WITH repeats
        j = rng.integers(0, len(up), len(up))
        boots.append(theta_hat(tl[i], tp[i], up[j]))
    boots = [b for b in boots if not np.isnan(b)]
    if not boots:                                         # judge too weak to correct
        return theta, np.nan, np.nan
    L, U = np.percentile(boots, [2.5, 97.5])
    return theta, L, U

In [ ]:
unlabeled_preds = [judge(t) for t in data["unlabeled_texts"]]
print("Raw judge pass rate (biased):", np.mean(unlabeled_preds))

theta, L, U = estimate_success_rate(test_human, test_preds, unlabeled_preds, B=2000)
print(f"Corrected success rate: {theta:.2f}   95% CI [{L:.2f}, {U:.2f}]")

target = 0.85
print("Ship?", "yes, lower bound passes the target" if L >= target else "not sure: lower bound is below the target")

## 7. Metrics for several outputs at once

In [ ]:
# Pass@k : at least one of k attempts is correct
attempts = data["pass_at_k_attempts"]
print("Pass@5:", any(attempts))

# Precision@k and Recall@k
ex = data["retrieval_example"]
top5 = ex["top5_is_relevant"]
print("Precision@5:", sum(top5) / len(top5))
print("Recall@5   :", sum(top5) / ex["total_relevant_in_corpus"])

In [ ]:
# Average pairwise similarity: lower = more diverse suggestions
def avg_pairwise_similarity(texts):
    v = embed(texts)
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    sim = v @ v.T
    return sim[np.triu_indices_from(sim, k=1)].mean()

print("Diverse    :", round(avg_pairwise_similarity(data["suggestions_diverse"]), 2))
print("Repetitive :", round(avg_pairwise_similarity(data["suggestions_repetitive"]), 2))